# MMELON Fine-Tuning

Fine-tune the pretrained MMELON multi-view model (`ibm-research/biomed.sm.mv-te-84m`) for
**binary classification** of molecules (e.g. hit / non-hit) from SMILES.

**Environment:** run with the `biomed-multiview` kernel (see [installation-troubleshooting.md](installation-troubleshooting.md)).

**Input:** CSV files with a SMILES column and a 0/1 label column.

**Output** (in `biomed-multi-view/output/`, which is gitignored):
```
output/
├── base_models/biomed.sm.mv-te-84m/     # pretrained model from Hugging Face (downloaded once)
└── finetuned/<RUN_NAME>_<timestamp>/
    ├── data/                            # train/val/test CSVs used for this run
    ├── checkpoints/                     # best checkpoints + last.ckpt
    ├── logs/                            # metrics.csv per epoch
    ├── config.json                      # settings used for this run
    ├── test_metrics.json
    └── test_predictions.csv
```

## Steps
1. Inputs and settings
2. Prepare data
3. Download the base model
4. Data loaders
5. Build the model
6. Train
7. Evaluate on the test set

## 1. Inputs and settings

**Edit these before running.**

In [ ]:
from pathlib import Path

# ── Data ──────────────────────────────────────────────────────────────────────
DATA_DIR = Path.home() / "Datasets" / "temp-sample"

TRAIN_PATH = DATA_DIR / "data_train.csv"
TEST_PATH = DATA_DIR / "data_test.csv"   # set to None to skip test evaluation
VAL_PATH = DATA_DIR / "data_val.csv"     # set to None to hold out VAL_FRACTION of train instead

SMILES_COL = "smiles"
LABEL_COL = "label"

# Name of this run; outputs go to output/finetuned/<RUN_NAME>_<timestamp>/
RUN_NAME = "temp-sample"

In [ ]:
import os

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model ID

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42

VAL_FRACTION = 0.1        # only used when VAL_PATH is None
MAX_PER_CLASS = None      # e.g. 100 for a quick test run; None uses all data
MONITOR = "val_rocauc"    # metric used to pick the best checkpoint ("val_rocauc" or "val_loss")
NUM_WORKERS = min(4, os.cpu_count())

In [ ]:
import json
from datetime import datetime

import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "output"
BASE_MODEL_DIR = OUTPUT_DIR / "base_models" / BASE_MODEL.split("/")[-1]
RUN_DIR = OUTPUT_DIR / "finetuned" / f"{RUN_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DATA_DIR = RUN_DIR / "data"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"

for path in (TRAIN_PATH, VAL_PATH, TEST_PATH):
    if path is not None:
        assert Path(path).exists(), f"File not found: {path}"

RUN_DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

config = dict(
    train_path=str(TRAIN_PATH), val_path=str(VAL_PATH) if VAL_PATH else None,
    test_path=str(TEST_PATH) if TEST_PATH else None,
    smiles_col=SMILES_COL, label_col=LABEL_COL, base_model=BASE_MODEL,
    epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, seed=SEED, val_fraction=VAL_FRACTION,
    max_per_class=MAX_PER_CLASS, monitor=MONITOR,
)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Repo root  : {REPO_ROOT}")
print(f"Base model : {BASE_MODEL_DIR}")
print(f"Run folder : {RUN_DIR}")
print(f"Device     : {'GPU' if torch.cuda.is_available() else 'CPU (training will be slow)'}")

## 2. Prepare data

The MMELON data pipeline reads `data_train.csv`, `data_val.csv` and `data_test.csv` from one folder,
with columns `smiles` and `label`. This step copies your files into the run's `data/` folder in that format.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


def load_split(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = {SMILES_COL, LABEL_COL} - set(df.columns)
    assert not missing, f"{path} is missing columns {missing}; found {df.columns.tolist()}"
    df = df[[SMILES_COL, LABEL_COL]].rename(columns={SMILES_COL: "smiles", LABEL_COL: "label"})
    n_before = len(df)
    df = df.dropna()
    df = df[df["smiles"].astype(str).str.strip() != ""]
    if len(df) < n_before:
        print(f"{path.name}: dropped {n_before - len(df)} rows with empty SMILES or label")
    df["label"] = df["label"].astype(int)
    assert set(df["label"].unique()) <= {0, 1}, f"{path} labels must be 0/1"
    return df.reset_index(drop=True)


def limit_per_class(df: pd.DataFrame, n: int) -> pd.DataFrame:
    parts = [g.sample(n=min(n, len(g)), random_state=SEED) for _, g in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)


splits = {"train": load_split(TRAIN_PATH)}
if VAL_PATH is not None:
    splits["val"] = load_split(VAL_PATH)
else:
    splits["train"], splits["val"] = train_test_split(
        splits["train"], test_size=VAL_FRACTION, stratify=splits["train"]["label"], random_state=SEED
    )
    print(f"No VAL_PATH: held out {VAL_FRACTION:.0%} of train as validation")
if TEST_PATH is not None:
    splits["test"] = load_split(TEST_PATH)

summary = []
for name, df in splits.items():
    if MAX_PER_CLASS is not None:
        df = limit_per_class(df, MAX_PER_CLASS)
    splits[name] = df.reset_index(drop=True)
    splits[name].to_csv(RUN_DATA_DIR / f"data_{name}.csv", index=False)
    summary.append(dict(split=name, samples=len(df), positives=int(df["label"].sum()),
                        positive_pct=round(100 * df["label"].mean(), 2)))

pd.DataFrame(summary)

## 3. Download the base model

Downloaded once into `output/base_models/`; later runs reuse it.

In [ ]:
from huggingface_hub import snapshot_download

if (BASE_MODEL_DIR / "model.safetensors").exists():
    print(f"Using existing base model: {BASE_MODEL_DIR}")
else:
    print(f"Downloading {BASE_MODEL} ...")
    snapshot_download(repo_id=BASE_MODEL, local_dir=BASE_MODEL_DIR)
    print(f"Saved to: {BASE_MODEL_DIR}")

## 4. Data loaders

Each SMILES is converted into the three MMELON views: text tokens, a 2D image and a molecular graph.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.WARNING)

import pytorch_lightning as pl
from torch.utils.data import DataLoader

from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel
from bmfm_sm.core.data_modules.namespace import LateFusionStrategy, TaskType
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule

pl.seed_everything(SEED)

dataset_args = {
    "task_type": TaskType.CLASSIFICATION,
    "num_tasks": 1,
    "modalities": ["TEXT_MODEL", "IMAGE_MODEL", "GRAPH_2D_MODEL"],
    "smiles_col": "smiles",
    "label_cols": ["label"],
    "split_col": None,
}


def make_loader(stage: str, shuffle: bool) -> DataLoader:
    dataset = MultiModalFinetuneDataPipeline(
        data_dir=str(RUN_DATA_DIR), dataset_args=dataset_args, stage=stage
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=dataset.collate_fn, num_workers=NUM_WORKERS)


train_loader = make_loader("train", shuffle=True)
val_loader = make_loader("val", shuffle=False)
test_loader = make_loader("test", shuffle=False) if "test" in splits else None

for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    if loader is not None:
        print(f"{name:5s}: {len(loader.dataset):6,} samples, {len(loader):5,} batches")

## 5. Build the model

Creates the fine-tuning model (three encoders + attention fusion + MLP prediction head) and loads the
pretrained encoder weights. The prediction head starts from random weights.

In [ ]:
FUSION_STRATEGY = LateFusionStrategy.ATTENTIONAL

lightning_module = FineTuneLightningModule(
    base_model_class="bmfm_sm.predictive.modules.smmv_model.SmallMoleculeMultiView",
    model_params={
        "agg_arch": FUSION_STRATEGY.value[0],
        "agg_gate_input": FUSION_STRATEGY.value[1],
        "agg_weight_freeze": FUSION_STRATEGY.value[2],
        "inference_mode": False,
    },
    task_type="classification",
    num_tasks=1,
    checkpoint_path=None,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    finetuning_args={
        "weight_freeze": "unfrozen",
        "initialization": "default",
        "head_arch": "mlp",
        "use_norm": True,
        "head_dropout": 0.2,
    },
)

pretrained_model = SmallMoleculeMultiViewModel.from_pretrained(
    FUSION_STRATEGY, model_path=str(BASE_MODEL_DIR), huggingface=True
)
result = lightning_module.model.load_state_dict(pretrained_model.state_dict(), strict=False)
del pretrained_model

# strict=False would silently skip mismatched weights, so check explicitly
assert not result.missing_keys and not result.unexpected_keys, (
    f"Pretrained weights did not match: missing={result.missing_keys[:5]}, "
    f"unexpected={result.unexpected_keys[:5]}"
)
print("Pretrained weights loaded (all keys matched).")

## 6. Train

Keeps the 3 best checkpoints by `MONITOR` plus `last.ckpt`. Per-epoch metrics are written to `logs/.../metrics.csv`.

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

monitor_mode = "min" if MONITOR.endswith("loss") else "max"
checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename=f"mmelon-epoch{{epoch:02d}}-{MONITOR}{{{MONITOR}:.3f}}",
    auto_insert_metric_name=False,
    monitor=MONITOR,
    mode=monitor_mode,
    save_top_k=3,
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    callbacks=[checkpoint_callback],
    logger=CSVLogger(save_dir=RUN_DIR, name="logs"),
    default_root_dir=RUN_DIR,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
    enable_model_summary=True,
)

In [ ]:
%%time
trainer.fit(lightning_module, train_loader, val_loader)

print(f"\nBest checkpoint: {checkpoint_callback.best_model_path}")
print(f"Best {MONITOR}: {checkpoint_callback.best_model_score:.4f}")

In [ ]:
# Validation metrics per epoch
metrics = pd.read_csv(Path(trainer.logger.log_dir) / "metrics.csv")
val_cols = ["epoch"] + [c for c in metrics.columns if c.startswith("val_")]
metrics[val_cols].dropna(subset=[c for c in val_cols if c != "epoch"], how="all").reset_index(drop=True)

## 7. Evaluate on the test set

Uses the best checkpoint from training. Saves `test_metrics.json` and `test_predictions.csv`
(one row per test molecule with its predicted probability of being positive).

In [ ]:
if test_loader is None:
    print("No TEST_PATH given; skipping test evaluation.")
else:
    # Loads the best checkpoint into lightning_module before testing
    test_metrics = trainer.test(lightning_module, dataloaders=test_loader, ckpt_path="best")[0]
    (RUN_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
    print(json.dumps(test_metrics, indent=2))

In [ ]:
if test_loader is not None:
    lightning_module.eval()
    device = lightning_module.device
    scores = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
            logits, _ = lightning_module(batch)
            scores.extend(torch.sigmoid(logits).reshape(-1).cpu().tolist())

    predictions = splits["test"].copy()
    assert len(predictions) == len(scores), "Number of predictions does not match test rows"
    predictions["score"] = scores
    predictions["pred"] = (predictions["score"] > 0.5).astype(int)
    predictions.to_csv(RUN_DIR / "test_predictions.csv", index=False)
    print(f"Saved {len(predictions):,} predictions to {RUN_DIR / 'test_predictions.csv'}")
    display(predictions.sort_values("score", ascending=False).head(10))

In [ ]:
# Files produced by this run
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(RUN_DIR)):60s} {path.stat().st_size / 1e6:8.2f} MB")